# Generative Adversarial Network (GAN) for EEG Generation

For this section, we want to generate EEG of both classes in our data (MDDS and HS). We will seperate the data into these two classes, and do the same for both:

- Find out the general format for how the EEG files should look
    - How many channels
    - Do different files have different amounts of channels
    - What kind of data does each channel have, and how is it formatted
- Create a discriminator model
- Create a generative model

Some things to look at: 

- Is there a way to make the EEG files compatible with GAN model with torch from lecture (image data)?

- There may be fine-tuning in terms of making the neurons. Is there an optimal way to down/upsample for EEG data?
    - Is there an process to test different iterations?

The following code is used from the pipeline to do some analysis on the data:

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import mne

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [3]:

# The code to find the project root for the file path
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")


# These are the specific file paths for the project
PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "nm000114"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_features"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

# Get one specific eeg file loaded
def get_eeg_file(subject_id: str, condition: str):
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

PROJECT_ROOT: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-
RAW_DATA_DIR: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114
PROCESSED_DIR: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/processed_features
RESULTS_DIR: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/results


In [7]:
# Get a specific file
file_path = get_eeg_file("sub-HS1", "P300")

print(type(file_path))
print(file_path.exists())

# Load the data for the first file
raw = mne.io.read_raw_edf(file_path, preload=True)

# Gets turned into raw numpy array
data = raw.get_data()
print("Shape: ", data.shape) # 22 channels with 154880 values in each (A 2D array)

# Get the timepoints for channel 0 
channel_0 = data[0, :] 

# Gives us the values in channel 0
print(channel_0[10:], len(channel_0))


# 22 channels and 154880 timestamps

<class 'pathlib.PosixPath'>
True
Extracting EDF parameters from /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114/sub-HS1/eeg/sub-HS1_task-P300_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 154879  =      0.000 ...   604.996 secs...
Shape:  (22, 154880)
[7.65058366e-06 2.45018692e-06 3.15024033e-06 ... 1.67512779e-05
 1.08508278e-05 5.45041581e-06] 154880


In [13]:
# These are all the channels they should have in common
COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]

At this point, I want to know whether all the files have the same shape -- what their format is to see how I could generate them

In [19]:
edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

In [36]:
# Parser cell to read all EDF files and extract data and labels for modeling

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition from the filename
    Returns:
        - eyesClosed
        - eyesOpen
        - P300
    """

    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")

In [52]:
# metadata list to hold all parsed information 
rows = []

# Scan all files in the raw data directory to ensure we can access them
edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

for filepath in edf_files:
    subject_id = filepath.parts[-3]  # e.g. subHS1
    condition = parse_condition_from_filename(filepath)
    label = infer_label_from_subject(subject_id)

    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": label,
        "condition": condition,
        "filepath": str(filepath),
    })


# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)

metadata_df.head(190)

,patient_id,recording_id,label,condition,filepath
0,sub-HS1,sub-HS1_task-P300_eeg,0,P300,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
1,sub-HS1,sub-HS1_task-eyesClosed_eeg,0,eyesClosed,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
2,sub-HS1,sub-HS1_task-eyesOpen_eeg,0,eyesOpen,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
3,sub-HS10,sub-HS10_task-P300_eeg,0,P300,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
4,sub-HS10,sub-HS10_task-eyesClosed_eeg,0,eyesClosed,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
...,...,...,...,...,...
176,sub-MDDS7,sub-MDDS7_task-P300_eeg,1,P300,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
177,sub-MDDS7,sub-MDDS7_task-eyesClosed_eeg,1,eyesClosed,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
178,sub-MDDS8,sub-MDDS8_task-P300_eeg,1,P300,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...
179,sub-MDDS9,sub-MDDS9_task-eyesClosed_eeg,1,eyesClosed,/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_s...


In [ ]:
# Get all the eeg files 
def get_all_eegs(files):
    raws = []
    for file in files:
        raw = mne.io.read_raw_edf(file, preload=True)
        raws.append(raw)

    return raws

all_eegs = get_all_eegs(edf_files)
    

In [39]:
print(f"There are {len(all_eegs)} eeg files")

# Get the channels, timepoints, and sample frequency (how many samples per second)
n_channels = []
n_timepoints = []
sfreqs = []


for raw in all_eegs:
    sfreqs.append(raw.info['sfreq'])
    data = raw.get_data()
    n_channels.append(data.shape[0])
    n_timepoints.append(data.shape[1])


# Print a bunch of information to try to understand the structure of the data
print(f"The mean sample frequency is {np.mean(sfreqs)}")
print(f"The mean amount of channels is {np.mean(n_channels)}")
print(f"The mean amount of timepoints is {np.mean(n_timepoints)}")
print(f"The std amount of timepoints is {np.std(n_timepoints)}")

print(f"The unique sample frequencies is {set(sfreqs)}")
print(f"The unique number of channel counts are {set(n_channels)}")
print(f"The unique number of timepoints are {set(n_timepoints)}")

print(f"Max number of timepoint: {max(n_timepoints)}")
print(f"Min number of timepoint: {min(n_timepoints)}")


There are 181 eeg files
The mean sample frequency is 256.0
The mean amount of channels is 21.23756906077348
The mean amount of timepoints is 104664.39779005524
The std amount of timepoints is 39737.60344141525
The unique sample frequencies is {256.0}
The unique number of channel counts are {20, 22}
The unique number of timepoints are {154880, 76800, 96256, 89856, 77312, 156160, 77568, 155392, 49664, 162304, 78080, 155904, 164096, 61696, 168448, 78592, 164608, 156416, 160512, 155648, 74752, 164864, 156672, 91136, 160768, 46080, 79104, 161024, 157184, 75520, 161536, 79616, 79872, 161792, 75776, 76032, 162048, 153856, 166144, 76288, 81920, 76544, 162560, 154624, 158720, 162816, 77056, 163072, 48384, 175616, 163328, 163584}
Max number of timepoint: 175616
Min number of timepoint: 46080


This gives us a solid lay of the land. 

The sfreq is standardized to 256.

There tend to be only 20 or 22 unique channels, but there are more likely 22. This is maybe doable. 

However, there are a lot of different timepoints, and the difference between them is pretty large (pretty large std). When thinking of a GAN, it seems like the information is not just about forming one 'image' (one timepoint), but almost like a video (multiple timepoints together). This makes things a little complicated for data augmentation

Because of the variability between the different samples, and because this is my first time creating a GAN, I'm thinking of simply cutting all the GANs so that they're all the same length as the minimum timepoint (46080) and the minimum unique channel count (20). This way, the generation will be the same. This model will be a bit different from the original, but it may work ok?

Something to also potentially think about is getting just 1 timepoint, this would probably be very unsuccessful though since it may not be standardized at all

Here's what I'm thinking of doing. I want to seperate each sample into all of its possible chunks so that I have a bunch of data, and run GroupKFold on them. I first need to chunk the data.

In [35]:
def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    

In [34]:
def make_windows(raw, label, group_id):
    # This data will be in the (channels, time) format
    data_raw = raw.copy()

    # This makes sure the format is consistent
    data_raw.pick(COMMON_CHANNELS) 

    # get data
    data = data_raw.get_data()      # shape: (n_channels, n_time points)
    sfreq = int(data_raw.info['sfreq'])  # sampling frequency (e.g. 256 Hz)

    n_channels, n_timepoints = data.shape
    window_size = sfreq # 1 second

    X_chunks = []
    y_chunks = []
    groups = []

    # Get the number of windows without including the last one
    n_windows = n_timepoints // window_size

    for i in range(n_windows):
        start = i * window_size
        end = start + window_size

        window = data[:, start:end] # (channels, sfreq)

        # Flatten the features
        features = window.flatten() # (timepoints, flattened data)
        
        X_chunks.append(features)
        y_chunks.append(label)
        groups.append(group_id)

    return X_chunks, y_chunks, groups

all_eegs = get_all_eegs(edf_files)



# Loop through all raws and add data to 
for raw in all_eegs:
    label = 




Extracting EDF parameters from /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114/sub-HS1/eeg/sub-HS1_task-P300_eeg.edf...


Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 154879  =      0.000 ...   604.996 secs...
-1.1950911726558146e-05
